<a href="https://colab.research.google.com/github/misrori/ai/blob/2025/youtube_play_list_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Először telepítsük a szükséges csomagokat
!pip install yt-dlp --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.1/172.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.2 MB/s eta 0:00:00


In [3]:
playlist_url = "https://www.youtube.com/playlist?list=PLy2IrygFGne_Aq4hE-_kN40bbun34HZLM"


In [ ]:
import yt_dlp
import os
import pandas as pd

def download_youtube_audio(url, output_dir="downloads"):
    """
    YouTube videó vagy audió letöltése.

    Args:
        url (str): A YouTube videó URL-je
        audio_only (bool): Csak audió letöltése (True) vagy videó (False)
        output_dir (str): Kimeneti könyvtár

    Returns:
        list: A letöltött fájl(ok) elérési útja
    """
    # Könyvtár létrehozása, ha nem létezik
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Fájlnév előkészítése
    output_template = os.path.join(output_dir, '%(title)s.%(ext)s')

    # Letöltési beállítások
    ydl_opts = {
        'outtmpl': output_template,
        'quiet': False,
        'no_warnings': False
    }

    output_files = []

    # Tartalom típus szerinti beállítások

    ydl_opts.update({
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
    })
    print(f"Audió letöltése a(z) {url} címről a legjobb minőségben...")

    # Audió letöltése
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info_dict = ydl.extract_info(url, download=True)
        audio_file = ydl.prepare_filename(info_dict)
        audio_file = os.path.splitext(audio_file)[0] + '.mp3'
        print(f"Audió sikeresen letöltve: {audio_file}")
        output_files.append(audio_file)


    return output_files

def get_playlist_video_links(playlist_url):
    if "playlist" not in playlist_url:
        print("Ez nem egy lejátszható lista URL-je.")
        print('https://www.youtube.com/playlist?list=.............   ilyen kell!')
        return

    ydl_opts = {
        "quiet": True,
        "extract_flat": True,
        "force_generic_extractor": True,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(playlist_url, download=False)
        if "entries" in info:
            list_of_urls=  [{"url": entry["url"], "title":entry["title"]} for entry in info["entries"] if "url" in entry]
    return pd.DataFrame(list_of_urls)

video_links = get_playlist_video_links(playlist_url)

for link in video_links['url']:
    download_youtube_audio(link)


In [ ]:
# miután végzett becsomagolja és egy zippbe letölti

# create a zip from downloads folder
!zip -r downloads.zip downloads
# move the files from downloads folder to downloaded folder
!mkdir downloaded
!mv downloads/* downloaded/

# download the zip file
from google.colab import files
files.download('downloads.zip')

